## Initialisation

Ce notebook a pour objectif d'ingérer les données brutes dans la base PostgreSQL.

On commence par établir la connexion à la base, puis on charge les deux datasets sources :
- **releves_incidents** : journal des incidents machines (opérateur, sévérité, type de défaut, shift…)
- **telemetry** : relevés capteurs par machine (température, pression, tension, rotation, pièces produites)

In [2]:
import pandas as pd
from sqlalchemy import create_engine, text, URL

DB_USER = "indusense_user"
DB_PASSWORD = "ThEP@ssW0rd"
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "indusense_db"

url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)

engine = create_engine(url)

try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print(f"✅ Connexion à PostgreSQL réussie ({DB_HOST}:{DB_PORT}/{DB_NAME})")
except Exception as e:
    print(f"❌ Échec de connexion : {e}")

df_incidents = pd.read_csv("artifacts/ingestions/incidents/releves_incidents.csv")
print(f"✅ releves_incidents.csv chargé : {df_incidents.shape[0]} lignes, {df_incidents.shape[1]} colonnes")

df_telemetry = pd.read_csv("artifacts/ingestions/incidents/telemetry.csv")
print(f"✅ telemetry.csv chargé : {df_telemetry.shape[0]} lignes, {df_telemetry.shape[1]} colonnes")

✅ Connexion à PostgreSQL réussie (localhost:5432/indusense_db)
✅ releves_incidents.csv chargé : 1245 lignes, 18 colonnes
✅ telemetry.csv chargé : 135626 lignes, 7 colonnes


## Ingestion CSV → PostgreSQL

Insertion des deux datasets dans leurs tables bronze respectives. On utilise `if_exists="replace"` pour recréer les tables à chaque exécution.

In [3]:
from sqlalchemy.orm import Session
from sqlalchemy import delete
from models.bronze import BronzeRelevesIncidents, BronzeTelemetry

incidents = df_incidents.copy()
incidents["date"] = pd.to_datetime(incidents["date"].astype(str) + " " + incidents["time"].astype(str))
incidents = incidents.drop(columns=["time"])
incidents["valid_parsing"] = True
incidents["valid_error_reason"] = None

telemetry = df_telemetry.copy()
telemetry = telemetry.rename(columns={"timestamp": "date"})
telemetry["date"] = pd.to_datetime(telemetry["date"])
telemetry["valid_parsing"] = True
telemetry["valid_error_reason"] = None

# Ingestion bronze_releves_incidents
with Session(engine) as session:
    session.execute(delete(BronzeRelevesIncidents))
    session.add_all([BronzeRelevesIncidents(**row) for row in incidents.to_dict("records")])
    session.commit()
    print(f"✅ {len(incidents)} objets insérés dans 'bronze_releves_incidents'")

# Ingestion bronze_telemetry
with Session(engine) as session:
    session.execute(delete(BronzeTelemetry))
    session.add_all([BronzeTelemetry(**row) for row in telemetry.to_dict("records")])
    session.commit()
    print(f"✅ {len(telemetry)} objets insérés dans 'bronze_telemetry'")

✅ 1245 objets insérés dans 'bronze_releves_incidents'


/var/folders/mm/cpswg40n0rlchv7sztxnsrfr0000gp/T/ipykernel_81202/2626273403.py:28: SAWarning: Identity map already had an identity for (<class 'models.bronze.telemetry.BronzeTelemetry'>, ('MACH-08', Timestamp('2025-12-08 05:00:00')), None), replacing it with newly flushed object.   Are there load operations occurring inside of an event handler within the flush?
  session.commit()
/var/folders/mm/cpswg40n0rlchv7sztxnsrfr0000gp/T/ipykernel_81202/2626273403.py:28: SAWarning: Identity map already had an identity for (<class 'models.bronze.telemetry.BronzeTelemetry'>, ('MACH-08', Timestamp('2025-12-13 04:00:00')), None), replacing it with newly flushed object.   Are there load operations occurring inside of an event handler within the flush?
  session.commit()
/var/folders/mm/cpswg40n0rlchv7sztxnsrfr0000gp/T/ipykernel_81202/2626273403.py:28: SAWarning: Identity map already had an identity for (<class 'models.bronze.telemetry.BronzeTelemetry'>, ('MACH-13', Timestamp('2025-11-02 03:00:00')), 

✅ 135626 objets insérés dans 'bronze_telemetry'


## Bronze data validation

- **bronze_releves_incidents** : `valid_parsing = False` si `severity` hors [1, 5]
- **bronze_releves_incidents** : `valid_parsing = False` si une colonne `type_*` est NULL (non entier)
- **bronze_telemetry** : `valid_parsing = False` pour la 2ème occurrence de chaque doublon `machine_id` + `date`

In [4]:
from sqlalchemy import update, select, func, literal_column, or_
from sqlalchemy.orm import Session
from models.bronze import BronzeRelevesIncidents, BronzeTelemetry

TYPE_COLUMNS = [
    BronzeRelevesIncidents.type_surchauffe,
    BronzeRelevesIncidents.type_baisse_pression,
    BronzeRelevesIncidents.type_vibration,
    BronzeRelevesIncidents.type_bruit_mecanique,
    BronzeRelevesIncidents.type_surconsommation,
    BronzeRelevesIncidents.type_blocage_mecanique,
    BronzeRelevesIncidents.type_alarme_capteur,
    BronzeRelevesIncidents.type_arret_urgence,
    BronzeRelevesIncidents.type_defaut_qualite,
]

# Validation bronze_releves_incidents — severity hors [1, 5]
with Session(engine) as session:
    stmt = (
        update(BronzeRelevesIncidents)
        .where(or_(
            BronzeRelevesIncidents.severity < 1,
            BronzeRelevesIncidents.severity > 5,
        ))
        .values(valid_parsing=False, valid_error_reason="severity_hors_intervalle")
    )
    result = session.execute(stmt)
    session.commit()
    print(f"✅ {result.rowcount} lignes marquées dans 'bronze_releves_incidents' (severity_hors_intervalle)")

# Validation bronze_releves_incidents — colonnes type_* non entières (NULL)
with Session(engine) as session:
    stmt = (
        update(BronzeRelevesIncidents)
        .where(or_(*[col.is_(None) for col in TYPE_COLUMNS]))
        .values(valid_parsing=False, valid_error_reason="type_non_entier")
    )
    result = session.execute(stmt)
    session.commit()
    print(f"✅ {result.rowcount} lignes marquées dans 'bronze_releves_incidents' (type_non_entier)")

# Validation bronze_telemetry — doublons machine_id + date (2ème occurrence)
ctid = literal_column('ctid')

with Session(engine) as session:
    ranked_subq = (
        select(
            ctid.label('row_ctid'),
            func.row_number().over(
                partition_by=[BronzeTelemetry.machine_id, BronzeTelemetry.date],
                order_by=ctid
            ).label('rn')
        )
        .select_from(BronzeTelemetry)
        .subquery()
    )
    stmt = (
        update(BronzeTelemetry)
        .where(ctid.in_(
            select(ranked_subq.c.row_ctid).where(ranked_subq.c.rn > 1)
        ))
        .values(valid_parsing=False, valid_error_reason="doublon_machine_date")
    )
    result = session.execute(stmt)
    session.commit()
    print(f"✅ {result.rowcount} lignes marquées dans 'bronze_telemetry' (doublon_machine_date)")

✅ 0 lignes marquées dans 'bronze_releves_incidents' (severity_hors_intervalle)
✅ 0 lignes marquées dans 'bronze_releves_incidents' (type_non_entier)
✅ 1346 lignes marquées dans 'bronze_telemetry' (doublon_machine_date)
